## แกะทำข้อสอบเก่า Traverse on Ellipsoid

In [1]:
import numpy as np
import math

In [2]:
"""
%   Created on Thu Sep 19 13:21:19 2021
%   @author: Chaiyut Charoenphon
%   Course : Geodetic Survey 2108415
%   
  Ref:
        Geoscience Australia - Python Geodesy Package
        Geodesy Module
        https://github.com/GeoscienceAustralia/GeodePy/blob/master/geodepy/geodesy.py
"""
from pygeodesy.ellipsoidalKarney import LatLon
import pygeodesy
from pyproj import Geod
import pyproj
import pygeodesy.dms as dms
from math import degrees, radians, sqrt, sin, cos, tan, asin, acos, atan, atan2
from geodepy.constants import grs80, utm, wgs84
from geodepy.convert import geo2grid, grid2geo, angular_typecheck

############################################################
def dd2dms(dd, PREC=5):
    #arcsecond is 5-digit decimal fraction
    return dms.toDMS(dd,prec=PREC)
def dms2dd(dms_str,SEP=' '):
    return dms.parseDMS(dms_str, sep=SEP)
    
#คำนวณรัศมีโลก ณ Latitude โดยประมาณ
def radius (B):
    # input:  B is Latitude in Degree unit
    # output: (Approximate) radius in m. unit (Approximate)
    B = radians(B) #converting into radians
    # WGS84
    a = 6378.137  #Radius at sea level at equator in m. unit
    b = 6356.752314245  #Radius at poles in m. unit
    c = (a**2*cos(B))**2
    d = (b**2*sin(B))**2
    e = (a*cos(B))**2
    f = (b*sin(B))**2
    R = sqrt((c+d)/(e+f))*1000.0
    return R

#แปลงค่าพิกัด xyz เป็น ละติจูด ลองจิจูด และ ความสูงเหนือทรงรี  หน่วยดีกรี
def xyz2lla(x,y,z):
    ecef = '+proj=geocent +ellps=WGS84' # Cartisian   
    lla = "epsg:4326" # WGS84 Geodetic    
    transproj = pyproj.Transformer.from_crs(ecef,lla)
    lat,lon,ell_h = transproj.transform(x,y,z,radians=False)
    return lat,lon,ell_h

#แปลงค่าพิกัด ละติจูด ลองจิจูด และ ความสูงเหนือทรงรี   เป็นพิกัด UTM Zone 47N
def lla2utm(lat,lon,ell_h,zone):
    '''
        I = lat ,long unit Degree
        I = ell_h unit metre
        O = E,N,h unit metre
    '''
    lla = "epsg:4326" # WGS84 Geodetic
    utm = "epsg:326" + zone # WGS84 UTM Zone 47N  
    transproj = pyproj.Transformer.from_crs(lla,utm,always_xy=True) # always_xy หมายถึงให้เรียง lon,lat
    E,N,h = transproj.transform(lon,lat,ell_h,radians=False)
    return E,N,h

#คำนวนสเกลแฟกเตอร์ในระบบพิกัด UTM โดยระบบพิกัดละติจูด ลองจิจูด และ โซน
def utm_scale_factor(lat,lon,zone):
    '''
        I = lat ,long unit Degree
        I = zone (UTM)
        O = scaleX, scaleY, scaleArea, mer_conv
    '''
    prj = pyproj.Proj('epsg:326' + zone)
    fc = prj.get_factors(lon,lat,radians=False)
    scaleX = fc.parallel_scale
    scaleY = fc.meridional_scale
    scaleArea = fc.areal_scale 
    mer_conv = fc.meridian_convergence
    return scaleX, scaleY, scaleArea, mer_conv

#คำนวนความสูงเหนือระดับน้ำทะเลปานกลาง อ้างอิงแบบจำลอง TGM2017 
def MSL(lat,lon,h):
    #  Download: tgm2017-1_2.pgm size 469 MB
    #  https://www.priabroy.name/archives/sdm_downloads/tgm2017-1
    geoid_model = r"C:\Lab_Geodetic_Survey\Lab4\tgm2017-1_2.pgm"
    geoid_interpolator = pygeodesy.GeoidKarney(geoid_model)
    # Get the geoid height
    pos = LatLon(lat, lon)
    N = geoid_interpolator(pos)
    return h-N

#แปลงค่าพิกัด xyz เป็นพิกัด Lambert Conformal Conic (LCC)
def xyz2LCC(x,y,z):   
    #Lambert Conformal Conic 1 standard parallel Conversion
    llc_1sp   = '+proj=lcc +lon_0=99 +lat_0=0 +x_0=500000 +y_0=0 +ellps=WGS84 +lat_1=33'
    #Lambert Conformal Conic conversion with scale factor
    llc_2sp_1 = '+proj=lcc +lon_0=99 +lat_0=0 +x_0=500000 +y_0=0 +ellps=WGS84 +k=0.9996'
    #Lambert Conformal Conic 2 standard parallel Conversion
    llc_2sp_2 = '+proj=lcc +lon_0=99 +lat_0=0 +x_0=500000 +y_0=0 +ellps=WGS84 +lat_1=9 +lat_2=12'
    
    ecef = {"proj":'geocent', "ellps":'WGS84'}
    transproj = pyproj.Transformer.from_crs(ecef,llc_1sp)
    E,N,h = transproj.transform(x,y,z)
    return (E,N,h)

def utm2lla(E,N,h,zone):   
    utm = "epsg:326" + zone # WGS84 UTM Zone 47N  
    lla = "epsg:4326" # WGS84 Geodetic 
    transproj = pyproj.Transformer.from_crs(utm,lla) # always_xy หมายถึงให้เรียง lon,lat
    lat,lon,h = transproj.transform(E,N,h,radians=False)
    return lat,lon,h
    
def utm2xyz(E,N,h,zone):
    utm = "epsg:326" + zone # WGS84 UTM Zone 47N  
    ecef = {"proj":'geocent', "ellps":'WGS84'}   
    transproj = pyproj.Transformer.from_crs(utm,ecef,always_xy=True) # always_xy หมายถึงให้เรียง lon,lat
    x,y,z = transproj.transform(E,N,h)
    return x,y,z
def lla2xyz(lat,lon,h):
    lla = "epsg:4326" # WGS84 Geodetic
    ecef = {"proj":'geocent', "ellps":'WGS84'}
    transproj = pyproj.Transformer.from_crs(lla,ecef,always_xy=True) # always_xy หมายถึงให้เรียง lon,lat
    x,y,z = transproj.transform(lon,lat,h,radians=False)
    return E,N,h    
def xyz2utm(x,y,z,zone):
    utm = "epsg:326" + zone # WGS84 UTM Zone 47N  
    ecef = {"proj":'geocent', "ellps":'WGS84'}
    transproj = pyproj.Transformer.from_crs(ecef,utm) # always_xy หมายถึงให้เรียง lon,lat
    E,N,h = transproj.transform(x,y,z)
    return E,N,h
#+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
def line_sf(zone1, east1, north1, zone2, east2, north2,
            hemisphere='south', ellipsoid=grs80, projection=utm):
    """
    Computes Line Scale Factor for a pair of Transverse Mercator Coordinates
    Ref: Deakin 2010, Traverse Computations on the Ellipsoid and on the
    Universal Transverse Mercator Projection, pp 35
    http://www.mygeodesy.id.au/documents/Trav_Comp_V2.1.pdf
    :param zone1: Station 1 Zone Number - 1 to 60
    :param east1: Station 1 Easting (m, within 3330km of Central Meridian)
    :param north1: Station 1 Northing (m, 0 to 10,000,000m)
    :param zone2: Station 2 Zone Number - 1 to 60
    :param east2: Station 2 Easting (m, within 3330km of Central Meridian)
    :param north2: Station 2 Northing (m, 0 to 10,000,000m)
    :param hemisphere: String - 'North' or 'South'(default)
    :param ellipsoid: Ellipsoid Object (default GRS80)
    :param projection: Projection Object (default Universal Transverse Mercator)
    :return: Line Scale Factor (relative to Zone 1 if different zones
    are entered)
    """
    # Re-project cross-zone coordinate to same UTM zone
    if zone1 != zone2:
        # Re-project Station 2 Coordinates to same zone as Station 1
        stn2_geo = grid2geo(zone2, east2, north2, hemisphere, ellipsoid)
        stn2_zone1 = geo2grid(stn2_geo[0], stn2_geo[1], zone1, ellipsoid)
        zone2 = stn2_zone1[1]
        east2 = stn2_zone1[2]
        north2 = stn2_zone1[3]

    # Comute easting distances from Central Meridian
    eastofcm1 = east1 - projection.falseeast
    eastofcm2 = east2 - projection.falseeast

    # Compute Mean Latitude
    lat1 = grid2geo(zone1, east1, north1, hemisphere, ellipsoid)[0]
    lat2 = grid2geo(zone2, east2, north2, hemisphere, ellipsoid)[0]
    lat_mean = (lat1 + lat2) / 2

    # Compute Line Scale Factor (Deakin 2010 Eq. 13)
    r_sq_m = (rho(lat_mean, ellipsoid) *
              nu(lat_mean, ellipsoid) *
              projection.cmscale ** 2)
    k1 = ((eastofcm1 ** 2 + eastofcm1 * eastofcm2 + eastofcm2 ** 2) /
          (6 * r_sq_m))
    k2 = ((eastofcm1 ** 2 + eastofcm1 * eastofcm2 + eastofcm2 ** 2) /
          (36 * r_sq_m))
    return projection.cmscale * (1 + k1 * (1 + k2))
    
def rho(lat, ellipsoid=grs80):
    """
    symbol = rho
    Return the radius of curvature of the ellipsoid in the meridian plane
    (rho) at a given latitude
    :param lat: latitude in decimal degrees
    :param ellipsoid: Ellipsoid Object
    :return: rho at specified latitude
    """
    return ((ellipsoid.semimaj * (1 - ellipsoid.ecc1sq)) /
            (1 - ellipsoid.ecc1sq * (sin(radians(lat)) ** 2)) ** 1.5)

def nu(lat, ellipsoid=grs80):
    """
    symbol = v
    Return the radius of curvature of the ellipsoid in the prime vertical plane
    (nu) at a given latitude
    :param lat: latitude in decimal degrees
    :param ellipsoid: Ellipsoid Object
    :return: nu at specified latitude
    """
    return (ellipsoid.semimaj / sqrt(1 - ellipsoid.ecc1sq * (sin(radians(lat)) ** 2)))   

def RadiusOfNormSection(lat, az12, ellipsoid=grs80):
    """
    Return the radius of  point 1 of the normal section in the direction Az12
    :param lat: latitude in decimal degrees
    :param az12: Azimuth Point 1 to Point 2 in decimal degrees
    :param ellipsoid: Ellipsoid Object default grs80
    :return: radius at specified latitude in m unit
    nu = v
    """  
    _rho = rho(lat,ellipsoid)
    _nu  = nu(lat,ellipsoid)
    return (_rho*_nu) / (_rho*(sin(radians(az12))**2) + _nu*(cos(radians(az12)))**2)   
   
def Ground_to_Ell(R,D,h1,h2):
    '''
        Convert Ground distance to ellipsoidal distance in m unit
        input:
            D is the ground distance in m. unit
            h1 and h2 are the ellipsoidal height (m. unit) at P1 and P2, respectively.
        output:
            ellipsoidal distance m. unit
    '''
    dh = h2-h1 # approximate
    c = R*sqrt((D**2-dh**2)/((R+h1)*(R+h2)))
    #c = R*D/(R+(h1+h2)*0.5)
    #print(c)
    s = 2.0*R*asin(c/(2.0*R))
    return s


#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$


### Step 1 Find Az for control station on ellipsoid

#### ใช้ Vincenty's Invert Formula ในการหา Geodesic Az

In [3]:
geom = Geod(ellps='WGS84')
lat2 = dms2dd("13 31 32.84002")
lon2 = dms2dd("99 44 39.81003")

lat1 = dms2dd("13 32 10.08001")
lon1 = dms2dd("99 44 22.84001") 

az12,az21,D = geom.inv(lon1,lat1,lon2,lat2) # azimuth and distance
print(f"{dd2dms(az12)} {dd2dms(az21+360)} {D:.4f} m")

print(D)
print(az21+360)
print(az21+360+72)

155°58′08.07208″ 335°58′12.04257″ 1253.0630 m
1253.0630076842374
335.9700118247249
407.9700118247249


In [4]:
geom = Geod(ellps='WGS84')
lat6 = dms2dd("13 32 03.97585")
lon6 = dms2dd("99 49 01.99500")

lat5 = dms2dd("13 32 33.03943")
lon5 = dms2dd("99 48 14.09309") 

az56,az65,D = geom.inv(lon5,lat5,lon6,lat6) # azimuth and distance
print(f"{dd2dms(az56)} {dd2dms(az65+360)} {D:.4f} m")

print(D)
print(az56)
print(az56+72)

121°48′08.65294″ 301°48′19.86669″ 1694.7949 m
1694.7948644147107
121.8024035933267
193.8024035933267


### Step 2 Fuction for adjust coordinate of reference coordinate

In [5]:
# ข้อสอบให้ฟังก์ชันมาแล้ว
def rotate_coordinates(E0, N0, E1, N1, angle_degrees):
    angle_radians = np.radians(angle_degrees)
    cos_theta = np.cos(angle_radians)
    sin_theta = np.sin(angle_radians)
    dE = E1 - E0
    dN = N1 - N0
    nE = E0 + dE* cos_theta + dN * sin_theta
    nN = N0 + -dE * sin_theta + dN * cos_theta
    return nE, nN

adj_az = 73.0
E5, N5 = rotate_coordinates(580551.5668, 1495404.3135, 586986.9256, 1497274.1405, adj_az)
print(E5, N5)

584221.2080757867 1489796.833780758


### Step 3 Adjust Slope Distance on Terrain into Geodesic Distance

In [6]:
# หา Geodesic Hor. Angle (แปลง dms2dd)
sta2 = dms2dd("100 23 57.85")
sta3 = dms2dd("210 05 34.12")
sta4 = dms2dd("99 45 24.31")
sta5 = dms2dd("275 34 11.88")
print(sta2, sta3, sta4, sta5)


100.39940277777778 210.0928111111111 99.75675277777778 275.5699666666667


In [7]:
# sta 2
R = RadiusOfNormSection(dms2dd('13 31 32.84002'),335.97001 + 100.39940 - 360) # unit m. # จุดเริ่มต้น # az21 + hoor ang 123 = az23
D = 3234.653 # unit m. ระยะเอียง (Slope distance)
h1 = 70.345 # unit m. #Ellipsoidal height at P1
h2 = 110.243 # unit m. #Ellipsoidal height at P2
ell_distance = Ground_to_Ell(R,D,h1,h2)
ell_distance

3234.361167306442

In [8]:
# Az32
az23 = 335.97001 + 100.3994 - 360
az32 = az23 + 180
az32

256.36941

In [9]:
# sta 3
R = RadiusOfNormSection(dms2dd('13 31 32.84002'),az32 + 210.0928 - 360) # unit m. # จุดเริ่มต้น # az34
D = 2517.356 # unit m. ระยะเอียง (Slope distance)
h1 = 110.243 # unit m. #Ellipsoidal height at P1
h2 = 20.565 # unit m. #Ellipsoidal height at P2
ell_distance = Ground_to_Ell(R,D,h1,h2)
ell_distance

2515.7323641000658

In [10]:
# Az 43
az34 = az32 + 210.0928 - 360
az43 = az34 + 180
az43

286.46221

In [11]:
# sta 4
R = RadiusOfNormSection(dms2dd('13 31 32.84002'),az43 + 99.7568 - 360) # unit m. # จุดเริ่มต้น # az34
D = 2009.683 # unit m. ระยะเอียง (Slope distance)
h1 = 20.565 # unit m. #Ellipsoidal height at P1
h2 = 100.021 # unit m. #Ellipsoidal height at P2
ell_distance = Ground_to_Ell(R,D,h1,h2)
ell_distance

2008.0926082466146

### Step 4 Traverse on Ellipsoid

#### คิด Az ก่อน (ใช้ Az21 กับ Az56 ของพี่เขานะ จะได้เช็คว่าเลขตรงไหมได้)

#### จริงๆ Az ต้องใช้ inverse formula แต่เขาไม่ได้ให้พิกัด station 3 กับ station 4 มา

In [12]:
az21 = 408.9700118 - 360
az23 = az21 + 100.39940277777778
az23

149.36941457777777

In [13]:
az32 = az23 + 180
az32

329.3694145777778

In [14]:
az34 = az32 + 210.0928111111111 - 360
az34

179.4622256888889

In [15]:
az43 = az34 + 180
az43

359.4622256888889

In [16]:
az45  = az43 + 99.75675277777778 - 360
az45

99.21897846666667

In [17]:
az54 = az45 + 180
az54

279.21897846666667

In [18]:
az56 = az54 + 275.5699666666667 - 360
az56

194.7889451333333

In [19]:
az_misclosure = az56 - 121.802403
az_misclosure

72.9865421333333

#### หา Lat/Lon ใช้ Vincenty's Direct Formula (forward)

In [20]:
geom = Geod(ellps="WGS84")
lat = dms2dd('13 31 32.84002')
lon = dms2dd('99 44 39.81003')
az23 = dms2dd("149 22 9.89") # อย่าใช้ฟังก์ชันแปลงเด็จขาด เอา dd มากดเครื่องคิดเลข (อย่าเอา dd มาทำ dms)
gs23 = 3234.361
endlon, endlat, backaz = geom.fwd(lon, lat, az23, gs23)
print(f"{dd2dms(endlat)} {dd2dms(endlon)} {dd2dms(backaz+360)}")

13°30′02.27875″ 99°45′34.60681″ 329°22′22.69434″


In [21]:
geom = Geod(ellps="WGS84")
lat = dms2dd('13 30 02.27875')
lon = dms2dd('99 45 34.60681')
az34 = dms2dd("179 27 44.01")
gs34 = 2515.732
endlon, endlat, backaz = geom.fwd(lon, lat, az34, gs34)
print(f"{dd2dms(endlat)} {dd2dms(endlon)} {dd2dms(backaz+360)}")

13°28′40.42166″ 99°45′35.39189″ 359°27′44.19313″


In [22]:
geom = Geod(ellps="WGS84")
lat = dms2dd('13 28 40.42166')
lon = dms2dd('99 45 35.39189')
az45 = dms2dd("99 13 8.32")
gs45 = 2008.093
endlon, endlat, backaz = geom.fwd(lon, lat, az45, gs45)
print(f"{dd2dms(endlat)} {dd2dms(endlon)} {dd2dms(backaz+360)}")

13°28′29.95087″ 99°46′41.29607″ 279°13′23.67867″


#### แปลงเป็น Easting/Northing

In [23]:
def lla2utm(lat,lon,ell_h,zone):
    lla = "epsg:4326"
    utm = "epsg:326" + str(zone)
    transproj = pyproj.Transformer.from_crs(lla, utm, always_xy=True)
    E, N, h = transproj.transform(lon, lat, ell_h, radians = False)
    return E, N, h

In [24]:
lla_sta3 = lla2utm(dms2dd('13 30 02.27875'),dms2dd('99 45 34.60681'),110.243,'47')
lla_sta3

(582207.391328628, 1492627.1420655756, 110.243)

In [25]:
lla_sta4 = lla2utm(dms2dd('13 28 40.42166'),dms2dd('99 45 35.39189'),20.565,'47')
lla_sta4

(582238.7734098234, 1490112.4018295086, 20.565)

In [26]:
lla_sta5 = lla2utm(dms2dd('13 28 29.95087'),dms2dd('99 46 41.29607'),100.021,'47')
lla_sta5

(584221.2899333616, 1489796.9160498744, 100.021)

#### หา dE,dN ของ Station 5 (ใช้ E/N ตามของพี่)

In [27]:
diffE = 584221.290 - 584221.2081
diffE

0.08189999999012798

In [28]:
diffN = 1489796.9160498744 - 1489796.8338
diffN

0.08224987448193133

#### หา Misclosure, Sum_Dist, Ratio

In [29]:
misclosure = math.sqrt(0.0819**2 + 0.0822**2)
misclosure

0.11603641669751785

In [30]:
sum_dist = 3234.361 + 2515.732 + 2008.093
sum_dist

7758.186

In [31]:
Ratio_denominator = (sum_dist/misclosure)
Ratio_denominator
# แสดงว่า Ratio = 1:66860

66859.92398596668